# Clasificación de flores con una CNN (TensorFlow / Keras)

**Objetivo:** construir, entrenar y evaluar una Red Neuronal Convolucional que clasifique imágenes de flores en 5 clases, y dejar el modelo listo para usarse desde una app Streamlit.

> **Colab:** activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

In [ ]:
import os, sys, json
# --- Configuración para Google Colab: clona el repo para poder importar `src` ---
REPO_URL = "https://github.com/TU_USUARIO/cnn-flores.git"   # <-- CAMBIAR por la URL de tu repositorio
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone {REPO_URL} repo
    %cd repo
if os.path.basename(os.getcwd()) == "notebooks":   # ejecución local desde /notebooks
    os.chdir("..")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
from src import data, model as model_lib, evaluate

tf.keras.utils.set_random_seed(data.SEED)
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Selección y descripción del dataset

Se usa **TF Flowers** (publicado por el equipo de TensorFlow): ~3.670 fotografías a color de flores en 5 clases: `daisy`, `dandelion`, `roses`, `sunflowers` y `tulips`.

**Por qué este dataset:**
- Es un problema de clasificación multiclase real, con fotos en condiciones variadas (ángulo, luz, fondo).
- Tiene tamaño suficiente para entrenar una CNN desde cero en Colab en pocos minutos.
- Las imágenes tienen tamaños diferentes, lo que obliga a hacer un preprocesamiento real.
- Al ser un dataset moderado, el *Data Augmentation* es realmente necesario para reducir el sobreajuste.

In [ ]:
data_dir = data.download_dataset("data")
paths, labels, class_names = data.list_images(data_dir)
print("Clases:", class_names)
print("Total de imágenes:", len(paths))

counts = pd.Series(labels).map(dict(enumerate(class_names))).value_counts().sort_index()
display(counts.to_frame("imágenes"))
counts.plot(kind="bar", color="steelblue", title="Imágenes por clase", rot=0)
plt.ylabel("Cantidad"); plt.show()

## 2. Exploración y visualización de las imágenes

In [ ]:
fig, axes = plt.subplots(len(class_names), 5, figsize=(14, 3 * len(class_names)))
rng = np.random.default_rng(1)
for r, c in enumerate(class_names):
    idx = rng.choice(np.where(labels == r)[0], 5, replace=False)
    for k, i in enumerate(idx):
        axes[r, k].imshow(Image.open(paths[i])); axes[r, k].axis("off")
    axes[r, 0].set_title(c, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Tamaños originales (muestra de 400 imágenes) para justificar el redimensionado
sample = rng.choice(len(paths), 400, replace=False)
sizes = np.array([Image.open(paths[i]).size for i in sample])
print(f"Ancho:  min={sizes[:,0].min()}  media={sizes[:,0].mean():.0f}  max={sizes[:,0].max()}")
print(f"Alto:   min={sizes[:,1].min()}  media={sizes[:,1].mean():.0f}  max={sizes[:,1].max()}")
plt.scatter(sizes[:,0], sizes[:,1], alpha=.4); plt.xlabel("Ancho"); plt.ylabel("Alto")
plt.title("Las imágenes NO tienen un tamaño uniforme"); plt.show()

## 3. Preprocesamiento, normalización y división de datos

- **Redimensionado** a 180×180 píxeles (tamaño único para la red).
- **Normalización** de los píxeles de [0, 255] a [0, 1] mediante una capa `Rescaling` *dentro* del modelo. Así la app Streamlit solo debe pasar la imagen redimensionada y se evita cualquier desajuste entre entrenamiento e inferencia.
- **División estratificada** (mantiene la proporción de clases): **70 % entrenamiento / 15 % validación / 15 % prueba**. El conjunto de prueba no se toca hasta la evaluación final.

In [ ]:
IMG_SIZE = (180, 180)
BATCH_SIZE = 32
ds, splits, class_names = data.build_datasets(data_dir, IMG_SIZE, BATCH_SIZE)

rows = []
for name, (p, y) in splits.items():
    row = {"conjunto": name, "total": len(p)}
    row.update({c: int((y == i).sum()) for i, c in enumerate(class_names)})
    rows.append(row)
display(pd.DataFrame(rows).set_index("conjunto"))

x, y = next(iter(ds["train"]))
print("Batch:", x.shape, x.dtype, "| rango píxeles:", float(x.numpy().min()), "-", float(x.numpy().max()))

## 4. Data Augmentation

In [ ]:
aug = model_lib.build_augmentation()
img = x[:1]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(img[0].numpy().astype("uint8")); axes[0, 0].set_title("Original")
for ax in axes.ravel()[1:]:
    ax.imshow(aug(img, training=True)[0].numpy().clip(0, 255).astype("uint8"))
    ax.set_title("Aumentada")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Construcción de la CNN

Arquitectura: *Data Augmentation → Rescaling → 4 bloques [Conv-BN-ReLU ×2 + MaxPool] (32→256 filtros) → GlobalAveragePooling → Dense(128) → Dropout(0.5) → Softmax(5)*.
`BatchNormalization`, `Dropout` y el aumento de datos se utilizan para controlar el sobreajuste.

In [ ]:
model = model_lib.build_cnn(IMG_SIZE + (3,), len(class_names))
model.summary()

## 6. Entrenamiento

In [ ]:
os.makedirs("models", exist_ok=True); os.makedirs("reports", exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("models/cnn_flores.keras", monitor="val_accuracy", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]
history = model.fit(ds["train"], validation_data=ds["val"], epochs=40, callbacks=callbacks)

## 7. Gráficas del entrenamiento

In [ ]:
evaluate.plot_history(history, "reports/training_curves.png"); plt.show()

## 8. Evaluación en el conjunto de prueba

Se usan **accuracy**, **precision, recall y F1 por clase** (más informativas que el accuracy solo) y la **matriz de confusión**.

In [ ]:
test_loss, test_acc = model.evaluate(ds["test"], verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}\n")
y_true, y_pred, probs = evaluate.get_predictions(model, ds["test"])
print(evaluate.text_report(y_true, y_pred, class_names))

In [ ]:
evaluate.plot_confusion_matrix(y_true, y_pred, class_names, "reports/confusion_matrix.png"); plt.show()

## 9. Predicciones con imágenes (12 imágenes del conjunto de prueba)

In [ ]:
p_te, y_te = splits["test"]
evaluate.show_predictions(model, p_te, y_te, class_names, n=12, img_size=IMG_SIZE,
                          path="reports/predictions.png"); plt.show()

In [ ]:
# Errores más seguros del modelo (ayuda a entender sus fallos)
wrong = np.where(y_true != y_pred)[0]
worst = wrong[np.argsort(-probs[wrong].max(axis=1))][:6]
fig, axes = plt.subplots(1, len(worst), figsize=(3.2 * len(worst), 3.6))
for ax, i in zip(np.atleast_1d(axes), worst):
    ax.imshow(Image.open(p_te[i])); ax.axis("off")
    ax.set_title(f"Real: {class_names[y_true[i]]}\nPred: {class_names[y_pred[i]]} ({probs[i].max():.0%})",
                 color="red", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Guardado del modelo

In [ ]:
model.save("models/cnn_flores.keras")
with open("models/class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f)
with open("reports/history.json", "w") as f:
    json.dump(history.history, f)
with open("reports/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(evaluate.text_report(y_true, y_pred, class_names))
print("Modelo guardado en models/cnn_flores.keras")

# Comprobación: recargar y predecir con el módulo que usa Streamlit
from src.predict import load_artifacts, predict_image
m2, names = load_artifacts()
print(predict_image(m2, Image.open(p_te[0]), names))

In [ ]:
# En Colab: descarga los artefactos para subirlos al repositorio
if "google.colab" in sys.modules:
    !zip -r artefactos.zip models reports
    from google.colab import files
    files.download("artefactos.zip")

## 11. Conclusiones

- Resume aquí la *accuracy* de prueba obtenida y las clases con mejor/peor F1.
- Comenta las confusiones más frecuentes de la matriz (p. ej. `roses` ↔ `tulips`) y su posible causa.
- Menciona posibles mejoras: *transfer learning* (MobileNetV2/EfficientNet), más datos, ajuste de hiperparámetros.